# ETL - Coffee Quality Database

Este notebook le o CSV, seleciona nove campos, limpa os valores e gera os fatos da base Prolog.

In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

In [2]:
RAIZ = Path.cwd()
ARQUIVO_ENTRADA = RAIZ / 'data' / 'raw' / 'arabica_data_cleaned.csv'
ARQUIVO_SAIDA = RAIZ / 'prolog' / 'base_conhecimento.pl'

## 1. Leitura e exploracao dos dados

In [3]:
dados = pd.read_csv(ARQUIVO_ENTRADA)
print(f'Linhas: {len(dados)}')
print(f'Colunas: {len(dados.columns)}')
dados.head()

Linhas: 1311
Colunas: 44


,Unnamed: 0,Species,Owner,Country.of.Origin,Farm.Name,Lot.Number,Mill,ICO.Number,Company,Altitude,...,Color,Category.Two.Defects,Expiration,Certification.Body,Certification.Address,Certification.Contact,unit_of_measurement,altitude_low_meters,altitude_high_meters,altitude_mean_meters
0,1,Arabica,metad plc,Ethiopia,metad plc,NaN,metad plc,2014/2015,metad agricultural developmet plc,1950-2200,...,Green,0,"April 3rd, 2016",METAD Agricultural Development plc,309fcf77415a3661ae83e027f7e5f05dad786e44,19fef5a731de2db57d16da10287413f5f99bc2dd,m,1950.0,2200.0,2075.0
1,2,Arabica,metad plc,Ethiopia,metad plc,NaN,metad plc,2014/2015,metad agricultural developmet plc,1950-2200,...,Green,1,"April 3rd, 2016",METAD Agricultural Development plc,309fcf77415a3661ae83e027f7e5f05dad786e44,19fef5a731de2db57d16da10287413f5f99bc2dd,m,1950.0,2200.0,2075.0
2,3,Arabica,grounds for health admin,Guatemala,"san marcos barrancas ""san cristobal cuch",NaN,NaN,NaN,NaN,1600 - 1800 m,...,NaN,0,"May 31st, 2011",Specialty Coffee Association,36d0d00a3724338ba7937c52a378d085f2172daa,0878a7d4b9d35ddbf0fe2ce69a2062cceb45a660,m,1600.0,1800.0,1700.0
3,4,Arabica,yidnekachew dabessa,Ethiopia,yidnekachew dabessa coffee plantation,NaN,wolensu,NaN,yidnekachew debessa coffee plantation,1800-2200,...,Green,2,"March 25th, 2016",METAD Agricultural Development plc,309fcf77415a3661ae83e027f7e5f05dad786e44,19fef5a731de2db57d16da10287413f5f99bc2dd,m,1800.0,2200.0,2000.0
4,5,Arabica,metad plc,Ethiopia,metad plc,NaN,metad plc,2014/2015,metad agricultural developmet plc,1950-2200,...,Green,2,"April 3rd, 2016",METAD Agricultural Development plc,309fcf77415a3661ae83e027f7e5f05dad786e44,19fef5a731de2db57d16da10287413f5f99bc2dd,m,1950.0,2200.0,2075.0


## 2. Selecao e limpeza dos campos

In [4]:
colunas_texto = ['Country.of.Origin', 'Variety', 'Processing.Method']
colunas_numericas = ['Aroma', 'Flavor', 'Acidity', 'Body', 'Total.Cup.Points']
cafes = dados[colunas_texto + colunas_numericas].copy()

for coluna in colunas_numericas:
    cafes[coluna] = pd.to_numeric(cafes[coluna], errors='coerce')

cafes = cafes.dropna(subset=colunas_numericas)
cafes[colunas_texto] = cafes[colunas_texto].fillna('desconhecido')
print(f'Linhas validas: {len(cafes)}')
print(f'Linhas ignoradas: {len(dados) - len(cafes)}')

Linhas validas: 1311
Linhas ignoradas: 0


## 3. Conversao para fatos Prolog

In [5]:
def normalizar_atomo(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', 'ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    return texto or 'desconhecido'

In [6]:
fatos = []

for identificador, (_, cafe) in enumerate(cafes.iterrows(), start=1):
    pais = normalizar_atomo(cafe['Country.of.Origin'])
    variedade = normalizar_atomo(cafe['Variety'])
    processo = normalizar_atomo(cafe['Processing.Method'])

    fato = (
        f"cafe({identificador}, {pais}, {variedade}, {processo}, "
        f"{cafe['Aroma']:.2f}, {cafe['Flavor']:.2f}, {cafe['Acidity']:.2f}, "
        f"{cafe['Body']:.2f}, {cafe['Total.Cup.Points']:.2f})."
    )
    fatos.append(fato)

fatos[:5]

['cafe(1, ethiopia, desconhecido, washed_wet, 8.67, 8.83, 8.75, 8.50, 90.58).',
 'cafe(2, ethiopia, other, washed_wet, 8.75, 8.67, 8.58, 8.42, 89.92).',
 'cafe(3, guatemala, bourbon, desconhecido, 8.42, 8.50, 8.42, 8.33, 89.75).',
 'cafe(4, ethiopia, desconhecido, natural_dry, 8.17, 8.58, 8.42, 8.50, 89.00).',
 'cafe(5, ethiopia, other, washed_wet, 8.25, 8.50, 8.50, 8.42, 88.83).']

## 4. Geracao da base de conhecimento

In [7]:
ARQUIVO_SAIDA.parent.mkdir(exist_ok=True)
cabecalho = '% Arquivo gerado automaticamente por etl.ipynb.\n% Nao edite manualmente.\n\n'
ARQUIVO_SAIDA.write_text(cabecalho + '\n'.join(fatos) + '\n', encoding='utf-8')

print(f'Fatos gerados: {len(fatos)}')
print(f'Arquivo gerado: {ARQUIVO_SAIDA.relative_to(RAIZ)}')

Fatos gerados: 1311
Arquivo gerado: prolog\base_conhecimento.pl
